# Match labelled rows with LLM-extracted rows
1. Load data
2. Vectorize rows
3. Match rows 
4. Compute accuracy


In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import geopy as gpy
import time
import itertools
import regex as re
from matplotlib import pyplot as plt
from src.data import *
from src.plot_functions import *
from src.post_process_functions import *
from src.geocoding import *
from src.hazard_def import *
from src.impact_def import *
from src.sanity_checks import *



## Load data

In [2]:
#load data (model)
post_processed = True #load or not post_processed data
model_name = "meta-llama/llama-4-scout-17b-16e-instruct"
nreports = 50
res_savename = f"post_processed_llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv" if post_processed else f"llm_response_impact_labelled_reports_{model_name.replace('/', '_')}.csv"
extracted_df = pd.read_csv(DATA_OUT_LLMS+res_savename)

#load data (labelled)
res_savename = "post_processed_labelled_reports_impacts_all.csv" if post_processed else "labelled_reports_impacts_all.csv"
labelled_df = pd.read_csv(DATA_OUT_LLMS+res_savename)


FileNotFoundError: [Errno 2] No such file or directory: '../Data_backup/results_llm/post_processed_llm_response_impact_labelled_reports_meta-llama_llama-4-scout-17b-16e-instruct.csv'

In [ ]:
#reformat output
num_cols = ["impactValue"]#"startYear", "startMonth", "startDay", "endYear", "endMonth", "endDay"
list_cols = ["country","location", "hazards", "impactsAnnotation"]
labelled_df = format_output(labelled_df, num_cols=num_cols, list_cols=list_cols)
extracted_df = format_output(extracted_df, num_cols=num_cols, list_cols=list_cols)
#labelled_df.replace(np.nan, None, inplace=True)
#extracted_df.replace(np.nan, None, inplace=True)
combined_df = pd.concat([labelled_df, extracted_df])


In [ ]:
combined_df[combined_df["appealCode"]=="MDRSV012"]

,reportDate,impactSubtype,impactValue,impactUnit,impactValuePrecision,impactValueMin,impactValueMax,impactsAnnotation,country,location,...,country_iso3_kw,hazards_reclass,impactValueOrig,impactUnitOrig,unit_type,country_kw,reportLink,disasterType,nathaz_text,impactType
106,2019-06-26,Affected People,NaN,people,NaN,NaN,NaN,[Situation analysis Description of the disaste...,[El Salvador],"[Morazn department, La Union department, El Br...",...,NaN,"['Tropical storm', 'Flood']",NaN,NaN,other,NaN,NaN,NaN,NaN,NaN
225,2019-06-26 00:00:00,Affected People,NaN,people,NaN,NaN,NaN,"[The rains have affected the entire country., ...",[El Salvador],"[eastern regions, San Miguel, La Unión departm...",...,SLV,"['Flood', 'Tropical storm']",NaN,people,other,El Salvador,https://adore.ifrc.org/Download.aspx?FileId=24...,Flood,['The major donors and partners of the Disaste...,NaN
226,2019-06-26 00:00:00,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,"[The rains have affected the entire country., ...",[El Salvador],"[eastern regions, San Miguel, La Unión departm...",...,SLV,"['Flood', 'Tropical storm']",NaN,NaN,other,El Salvador,https://adore.ifrc.org/Download.aspx?FileId=24...,Flood,['The major donors and partners of the Disaste...,NaN
227,2019-06-26 00:00:00,"Water, Sanitation, and Hygiene Infrastructure",NaN,undefined WASH facilities,NaN,NaN,NaN,"[The rains have affected the entire country., ...",[El Salvador],"[eastern regions, San Miguel, La Unión departm...",...,SLV,"['Flood', 'Tropical storm']",NaN,NaN,other,El Salvador,https://adore.ifrc.org/Download.aspx?FileId=24...,Flood,['The major donors and partners of the Disaste...,NaN


## Match
1. Vectorize columns that need to be compared using cosine similarity
2. Compute cosine similarity for those columns for each possible extracted-labelled pair
3. Add absolute difference of impactValue between each possible extracted-labelled pair.
    Need to consider NaN from not NaN separately. Only try matching non-NaNs with non-NaNs 
    (and nans with nan?)
4. Compute Intersection-Over-Union of polygons for each possible pair
5. Match by maximizing similarity and -impactvalu_idff and -IoT. Allow for more than one match.  


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def vectorize(cell_values, unique_values):
    """vectorizing function for categorical columns"""
    #cell_values = list() if not cell_values else cell_values
    cell_values = [cell_values] if not isinstance(cell_values, list) else cell_values
    vector = [1 if unique_value in cell_values else 0 for unique_value in unique_values]
    return np.array(vector)

def make_cosine_matrix():
    return

def split_nans(df,key):
    return df[~df[key].isna()], df[df[key].isna()]




unique_countries_ISO = [country.alpha_3 for country in pycountry.countries]
unique_country_names = [country.name for country in pycountry.countries]
pattern = '|'.join(map(re.escape, unique_country_names))

unique_dict = {#mapping dictonary
    'hazards' : hazard_main_types_emdat_extended,
    'country' : unique_country_names,
    'startYear' : np.arange(1980, 2025).tolist(),
    'startMonth' : np.arange(1, 13).tolist(),
    'startDay' : np.arange(1, 32).tolist(),
    'endYear' : np.arange(1980, 2025).tolist(),
    'endMonth' : np.arange(1, 13).tolist(),
    'endDay' : np.arange(1, 32).tolist(),
    'impactSubtype' : impactSubtype_list,
    'impactUnit' : combined_df.impactUnit.unique().tolist()
}

match_idxs = {}
for appeal, ext_group in extracted_df.groupby("appealCode"):
    matching_cols = list(unique_dict.keys())
    lab_group = labelled_df[labelled_df["appealCode"] == appeal]

    if lab_group.shape[0] == 0:
        continue

    ext_vect_df = pd.DataFrame(columns=matching_cols)
    lab_vect_df = pd.DataFrame(columns=matching_cols)

    #vectorize
    for col in matching_cols:
        ext_vect_df[col] = ext_group[col].apply(vectorize, unique_values=unique_dict[col])
        lab_vect_df[col] = lab_group[col].apply(vectorize, unique_values=unique_dict[col])

    #compute cosine distance
    matching_cols = matching_cols# + ["impactValue"]

    dist_mat = np.full((len(ext_vect_df), len(lab_vect_df), len(matching_cols)), np.nan)
    for k, col in enumerate(matching_cols):
        if col == "impactValue":
            pass
            # add negative absolute diff for impact value
            # dist must be negative as we want to maximize similarity
            #dist_mat[:,:,k] = -np.abs(ext_group["impactValue"].values.reshape(-1, 1) - lab_group["impactValue"].values.reshape(1, -1))
        else: #compute cosine similarity
            # Convert Series of arrays/lists to 2D numpy arrays
            X = np.stack(ext_vect_df[col].values) #nsamples, nfeatures
            Y = np.stack(lab_vect_df[col].values)
            # Compute cosine similarity
            dist_mat[:,:,k] = cosine_similarity(X, Y)

    #need to separate Nans from not Nans
    not_nan_ext_df, nan_ext_df = split_nans(ext_group,"impactValue")
    not_nan_lab_df, nan_lab_df = split_nans(lab_group,"impactValue")

    #retrieve positional indices
    nan_id_ext = nan_ext_df.reset_index().index.values
    nan_id_lab = nan_lab_df.reset_index().index.values
    not_nan_id_ext = not_nan_ext_df.reset_index().index.values
    not_nan_id_lab = not_nan_lab_df.reset_index().index.values

    #remove nans from dist matrix
    if len(nan_id_ext):
        if len(nan_id_lab):
            dist_mat_notna = np.delete(np.delete(dist_mat, nan_id_ext, axis=0), nan_id_lab, axis=1)
            dist_mat_na = dist_mat[nan_id_ext, :, :][:,nan_id_lab,:]
        else:
            dist_mat_notna = np.delete(dist_mat, nan_id_ext, axis=0)
            dist_mat_na = dist_mat[nan_id_ext, :, :]
    else:
        if len(nan_id_lab):
            dist_mat_notna = np.delete(dist_mat, nan_id_lab, axis=1)
            dist_mat_na = None
        else:
            dist_mat_notna = dist_mat
            dist_mat_na = None

    #calculate diff
    value_diff = -np.abs(not_nan_ext_df["impactValue"].values.reshape(-1, 1) - not_nan_lab_df["impactValue"].values.reshape(1, -1))
    dist_mat_notna = np.append(dist_mat_notna, value_diff[:,:, None], axis=2)

    ## TODO ADD IOTs

    #find possible candidates based on max similarity
    agg_sim_notna = np.nansum(dist_mat_notna, axis=2) #aggregate score for all matching columns (#ext, #lab)
    print(appeal)
    print(agg_sim_notna)
    max_sim_notna = np.max(agg_sim_notna, axis=1) #find max similarity value (#ext)
    id_match_ext_notna, id_match_lab_notna = np.where(agg_sim_notna == max_sim_notna[:,None]) #possible candidates with max similarity

    #reindex back in original df
    id_match_ext = not_nan_ext_df.iloc[id_match_ext_notna].index.values.flatten()
    id_match_lab = not_nan_lab_df.iloc[id_match_lab_notna].index.values.flatten()

    #find possible candidates based on max similarity
    if dist_mat_na is not None:
        agg_sim_na = np.nansum(dist_mat_na, axis=2) #aggregate score for all matching columns (#ext, #lab)
        max_sim_na = np.max(agg_sim_na, axis=1) #find max similarity value (#ext)
        id_match_ext_na, id_match_lab_na = np.where(agg_sim_na == max_sim_na[:,None]) #possible candidates with max similarity
        id_match_ext_na = nan_ext_df.iloc[id_match_ext_na].index.values.flatten()
        id_match_lab_na = nan_lab_df.iloc[id_match_lab_na].index.values.flatten()
        id_match_ext = np.append(id_match_ext, id_match_ext_na)
        id_match_lab = np.append(id_match_lab, id_match_lab_na)
    #write as df
    match_table = pd.DataFrame((id_match_ext, id_match_lab),
                               index = ["ext_match_id", "lab_match_id"]).T

    ##find possible candidates based on max similarity
    #agg_sim = np.nansum(dist_mat, axis=2) #aggregate score for all matching columns (#ext, #lab)
    #max_sim = np.max(agg_sim, axis=1) #find max similarity value (#ext)
    #id_match_ext, id_match_lab = np.where(agg_sim == max_sim[:,None]) #possible candidates with max similarity
    #
    #match_table = pd.DataFrame((id_match_ext, ext_group.iloc[id_match_ext].impactValue.values,
    #                           id_match_lab, lab_group.iloc[id_match_lab].impactValue.values),
    #                           index = ["ext_id", "ext_impactValue", "lab_id", "lab_impactValue"]).T
    #
    #for ext_id, group in match_table.groupby("ext_id"):
    #    value_diff = np.abs(group["lab_impactValue"] - group["ext_impactValue"] )
    #    if not pd.isna(group["ext_impactValue"]):
    #        #if value is not NaN, only match with numbers
    #        match_idxs[ext_id] = value_diff.dropna(how="any").min().lab_id
    #    else:
    #        #if value is NaN, only match with NaNs?
    #        match_idxs[ext_id] = group.where(group["lab_impactValue"].isna()).lab_id


MDRBD022
[[-5.42347600e+06 -6.59999700e+06 -5.49999700e+06 -7.58999700e+06
  -7.59984767e+06 -5.42347800e+06 -7.59600900e+06 -7.50142500e+06
  -7.56817900e+06 -7.59984967e+06]
 [-1.87651500e+06 -6.99997000e+05 -1.79999700e+06 -2.89997000e+05
  -2.99848670e+05 -1.87651600e+06 -2.96009000e+05 -2.01424000e+05
  -2.68179000e+05 -2.99849670e+05]
 [-2.17640100e+06 -9.99883000e+05 -2.09988300e+06 -9.88300000e+03
  -2.93300000e+01 -2.17640200e+06 -3.87100000e+03 -9.84530000e+04
  -3.17010000e+04 -3.03300000e+01]
 [-1.57651600e+06 -3.99995000e+05 -1.49999700e+06 -5.89997000e+05
  -5.99849670e+05 -1.57651400e+06 -5.96007000e+05 -5.01426000e+05
  -5.68179000e+05 -5.99849670e+05]
 [-2.17119600e+06 -9.94677000e+05 -2.09467500e+06 -4.67700000e+03
  -5.16967000e+03 -2.17119600e+06 -1.32900000e+03 -9.32480000e+04
  -2.64930000e+04 -5.16967000e+03]
 [-2.13651400e+06 -9.59997000e+05 -2.05999700e+06 -2.99970000e+04
  -3.98476700e+04 -2.13651600e+06 -3.60090000e+04 -5.85670000e+04
  -8.17900000e+03 -3.984

ValueError: zero-size array to reduction operation maximum which has no identity

In [65]:
match_idxs

[array([ 0,  0,  9,  0,  3,  7,  8,  4, 10,  0,  2,  6,  0,  0,  0,  2,  6,
         3,  7,  8,  0,  2,  6]),
 array([0, 0, 0, 5, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9]),
 array([ 0,  1,  2,  4,  5,  6,  7,  8,  9, 10, 16,  0,  1,  2,  4,  5,  6,
         7,  8,  9, 10, 16, 17,  0,  1,  2,  4,  5,  6,  7,  8,  9, 10, 16,
        17, 11, 12, 13,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12,
        13, 14, 15, 16, 17,  0,  1,  2,  4,  5,  6,  7,  8,  9, 10, 16, 17,
         0,  1,  2,  4,  5,  6,  7,  8,  9, 10, 16, 17,  0,  1,  2,  4,  5,
         6,  7,  8,  9, 10, 16, 17,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
        10, 11, 12, 13, 14, 15, 16, 17,  0,  1,  2,  4,  5,  6,  7,  8,  9,
        10, 16, 17]),
 array([ 0,  8,  1,  9,  2, 11,  4,  5, 12, 13, 15,  4,  5, 12, 13, 15,  6,
         7,  0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15,
         0,  8,  1,  9, 10,  2, 11,  4,  5, 12, 13, 15,  4,  5, 12, 13, 15,
         0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 

## Match

In [ ]:
#compute cosine distance
matching_cols = matching_cols + ["impactValue"]

dist_mat = np.full((len(ext_vect_df), len(lab_vect_df), len(matching_cols)), np.nan)
for k, col in enumerate(matching_cols):
    if col == "impactValue":
        # add negative absolute diff for impact value
        # dist must be negative as we want to maximize similarity
        dist_mat[:,:,k] = -np.abs(extracted_df["impactValue"].values.reshape(-1, 1) - labelled_df["impactValue"].values.reshape(1, -1))
    else: #compute cosine similarity
        # Convert Series of arrays/lists to 2D numpy arrays
        X = np.stack(ext_vect_df[col].values) #nsamples, nfeatures
        Y = np.stack(lab_vect_df[col].values)
        # Compute cosine similarity
        dist_mat[:,:,k] = cosine_similarity(X, Y)




In [ ]:
#find match idx from cosine matrix
match_idx = dist_mat.sum(axis=2).argmax(axis=1)

#join extracted and labelled dataframes
repeated_df = labelled_df.loc[np.repeat(match_idx, 1)].reset_index(drop=True)
repeated_df = repeated_df.add_suffix('_matched')
joined_df = pd.concat([extracted_df, repeated_df], axis=1)


In [ ]:
joined_df.sort_index(axis=1)

,appealCode,appealCode_matched,comments_matched,country,country_iso3,country_iso3_kw,country_iso3_kw_matched,country_iso3_matched,country_kw,country_matched,...,reportDate_matched,reportLink,startDay,startDay_matched,startMonth,startMonth_matched,startYear,startYear_matched,unit_type,unit_type_matched
0,MDR55001,MDRBD022,NaN,"[Kiribati, Papua New Guinea, Solomon Islands, ...",NaN,FJI,NaN,NaN,Fiji,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=16...,NaN,16.0,NaN,7.0,NaN,2019.0,other,other
1,MDR55001,MDRBD022,NaN,"[Kiribati, Papua New Guinea, Solomon Islands, ...",NaN,FJI,NaN,NaN,Fiji,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=16...,NaN,16.0,NaN,7.0,NaN,2019.0,other,other
2,MDR55001,MDRBD022,NaN,[Vanuatu],NaN,FJI,NaN,NaN,Fiji,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=16...,NaN,16.0,NaN,7.0,NaN,2019.0,other,other
3,MDR55001,MDRBD022,NaN,[Vanuatu],NaN,FJI,NaN,NaN,Fiji,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=16...,NaN,16.0,NaN,7.0,NaN,2019.0,other,other
4,MDR55001,MDRBD022,NaN,[Tuvalu],NaN,FJI,NaN,NaN,Fiji,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=16...,NaN,16.0,NaN,7.0,NaN,2019.0,other,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,MDRZM022,MDRBD022,NaN,[Zambia],NaN,ZMB,NaN,NaN,Zambia,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,18.0,NaN,7.0,2023.0,2019.0,other,other
271,MDRZM022,MDRBD022,NaN,[Zambia],NaN,ZMB,NaN,NaN,Zambia,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,18.0,NaN,7.0,2023.0,2019.0,other,other
272,MDRZM022,MDRBD022,NaN,[Zambia],NaN,ZMB,NaN,NaN,Zambia,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,16.0,NaN,7.0,2024.0,2019.0,other,other
273,MDRZM022,MDRBD022,NaN,[Zambia],NaN,ZMB,NaN,NaN,Zambia,[Bangladesh],...,2019-07-19,https://adore.ifrc.org/Download.aspx?FileId=84...,NaN,18.0,NaN,7.0,NaN,2019.0,other,other


In [ ]:
repeated_df

,matched_reportDate,matched_impactSubtype,matched_impactValue,matched_impactUnit,matched_impactValuePrecision,matched_impactValueMin,matched_impactValueMax,matched_impactsAnnotation,matched_country,matched_location,...,matched_endDay,matched_hazards,matched_appealCode,matched_comments,matched_country_iso3,matched_country_iso3_kw,matched_hazards_reclass,matched_impactValueOrig,matched_impactUnitOrig,matched_unit_type
0,2019-07-19,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,[It is also reported that embankments have bee...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",...,NaN,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],NaN,NaN,other
1,2019-07-19,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,[It is also reported that embankments have bee...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",...,NaN,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],NaN,NaN,other
2,2019-07-19,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,[It is also reported that embankments have bee...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",...,NaN,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],NaN,NaN,other
3,2019-07-19,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,[It is also reported that embankments have bee...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",...,NaN,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],NaN,NaN,other
4,2019-07-19,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,[It is also reported that embankments have bee...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",...,NaN,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],NaN,NaN,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,2019-07-19,Affected People,2176519.0,people,exact,NaN,NaN,[DREF operation n MDRBD022 Glide n FL-2019-000...,[Bangladesh],[],...,18.0,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],2176519.0,people,other
271,2019-07-19,Affected People,2176519.0,people,exact,NaN,NaN,[DREF operation n MDRBD022 Glide n FL-2019-000...,[Bangladesh],[],...,18.0,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],2176519.0,people,other
272,2019-07-19,Other Infrastructure Impacts,NaN,unknown,NaN,NaN,NaN,[It is also reported that embankments have bee...,[Bangladesh],"[Kurigram district, Gaibandha district, Lalmon...",...,NaN,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],NaN,NaN,other
273,2019-07-19,Affected People,2176519.0,people,exact,NaN,NaN,[DREF operation n MDRBD022 Glide n FL-2019-000...,[Bangladesh],[],...,18.0,[Flood],MDRBD022,NaN,NaN,NaN,['Flood'],2176519.0,people,other


In [ ]:
extracted_df

,impactSubtype,impactValue,impactUnit,impactValuePrecision,country,location,startYear,startMonth,startDay,endYear,...,reportLink,disasterType,nathaz_text,impactType,country_iso3,country_iso3_kw,hazards_reclass,impactValueOrig,impactUnitOrig,unit_type
0,Affected People,43880.0,people,exact,"[Kiribati, Papua New Guinea, Solomon Islands, ...",[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Tropical storm'],43880.0,people,other
1,Affected People,34573.0,people,exact,"[Kiribati, Papua New Guinea, Solomon Islands, ...",[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Tropical storm'],34573.0,people,other
2,Residential Buildings,900.0,homes,exact,[Vanuatu],[West Tanna],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Tropical storm'],900.0,houses,other
3,Human Health and Wellbeing,85.0,unknown,exact,[Vanuatu],[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Tropical storm'],85.0,per cent,other
4,"Access to Water, Sanitation, and Hygiene",414.0,people,exact,[Tuvalu],[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=16...,Cyclone,['He explained clearly what the indicators mea...,NaN,NaN,FJI,['Tropical storm'],138.0,households,other
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,Crop Production and Forestry,NaN,undefined crop production and forestry,NaN,[Zambia],[],2023.0,NaN,NaN,2024.0,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],NaN,NaN,other
271,Affected Livestock and Animals,NaN,undefined affected animals,NaN,[Zambia],[],2023.0,NaN,NaN,2024.0,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],NaN,NaN,other
272,Other Economic and Livelihood Impacts,11.0,CHF,exact,[Zambia],[],2024.0,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],11.0,CHF million,other
273,Access to Food,NaN,people,NaN,[Zambia],[],NaN,NaN,NaN,NaN,...,https://adore.ifrc.org/Download.aspx?FileId=84...,Drought,['SITUATION ANALYSIS Description of the crisis...,NaN,NaN,ZMB,['Drought'],NaN,NaN,other
